# **Notebook 1 - Getting Started**

In this notebook, you will learn the **fundamentals of SHOTpy**. By the end, you will be comfortable **defining and solving an optimization problem**.


## Table of Contents
1. [Installation](#1-installation)

2. [Creating Variables](#2-creating-variables)

3. [Your First Optimization Problem](#3-your-first-optimization-problem)

4. [Objectives](#4-objectives)

5. [Constraints](#5-constraints)

6. [Retrieving Solution Details](#6-retrieving-solution-details)

7. [Putting It All Together: A Full MINLP Example](#7-putting-it-all-together:-a-full-MINLP-example)

8. [Notebook 1 - Key Takeaways](#8-key-takeaways)


## **1. Installation**

First, import the SHOTpy module and create an environment.

In [1]:
import sys
import os
from pathlib import Path

# Find SHOTpy module (may need changing if your project structure is different)
base = Path.cwd()
repo_root = base.parent   # one level up; use .parent.parent for two levels, etc.
sys.path.insert(0, str(repo_root))  

import SHOTpy

# Create a SHOT solver and get its environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()
print("SHOT solver and environment created successfully!")

SHOT solver and environment created successfully!


**Current status:** SHOTpy is not yet available through `pip install shotpy`. To use SHOTpy, obtain the source code and ensure that the repository root is available on Python's module search path.

## **2. Creating Variables**

Variables are the fundamental building blocks of optimization problems. SHOTpy supports:
- **Real** (continuous) variables
- **Integer** variables
- **Binary** variables

To create a variable, use the following syntax:

`var_name = SHOTpy.Variable("name", index, SHOTpy.VariableType.<Type>, lower_bound, upper_bound)`

- `name` (str): Human-readable identifier shown by the solver and in prints.
- `index` (int): Unique internal index for the variable.
- `<Type>`: Variable kind — `Real`, `Integer`, or `Binary`.
- `lower_bound` (float): Minimum allowed value.
- `upper_bound` (float): Maximum allowed value.
- **Important:** Always add variable to problem by using `problem.addVariable(var)` 


### Example:

In [2]:
# Create a continuous variable: x ∈ [0, 10]
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)

# Create an integer variable: y ∈ {0, 1, 2, ..., 5}
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)

# Create a binary variable: z ∈ {0, 1}
z = SHOTpy.Variable("z", 2, SHOTpy.VariableType.Binary, 0.0, 1.0)

print(f"Created variables: {x.name}, {y.name}, {z.name}. Note, they are not added to a problem since we haven't created one yet!")

Created variables: x, y, z. Note, they are not added to a problem since we haven't created one yet!


## **3. Your First Optimization Problem**

Before exploring all the ways you can build objectives and constraints, let's solve something small and simple. Every SHOTpy problem follows the same structure:

| Item | Syntax | Purpose |
|---|---|---|
| **Create Solver** | `solver = SHOTpy.Solver()` | Initialize SHOT solver instance |
| **Get Environment** | `env = solver.getEnvironment()` | Access solver environment for problem creation |
| **Create Problem** | `problem = SHOTpy.Problem(env)` | Create optimization problem object, then add variables, an objective and constraints |
| **Finalize Problem** | `problem.finalize()` | Lock problem structure before solving |
| **Set Problem** | `solver.setProblem(problem)` | Assign problem to solver |
| **Solve** | `solver.solveProblem()` | Execute solver |
| **Get Results** | `solver.getPrimalSolution()` | Access primal solution |

Let's see this in action with a two-variable linear program:
$$
\begin{align*}
\text{minimize} \quad & 2x + 3y \\
\text{subject to} \quad & x + y \geq 5 \\
& 0 \leq x, y \leq 100
\end{align*}
$$

In [ ]:
# Create solver 
solver = SHOTpy.Solver()

# Get environment
env = solver.getEnvironment()

# Create problem
problem = SHOTpy.Problem(env)

# Variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Objective: minimize 2x + 3y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.LinearTerm(3.0, y))
problem.setObjective(objective)

# Constraint: x + y >= 5
constraint = SHOTpy.LinearConstraint(0, "c1", 5.0, SHOTpy.SHOT_DBL_MAX) # SHOTpy.SHOT_DBL_MAX is a constant representing infinity, see §5 for more details
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Finalize problem
problem.finalize()

# Set problem
solver.setProblem(problem)

# Solve
solver.solveProblem()

# Get results
sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Optimal objective value: 10.0000
Optimal solution: x = 5.0000, y = 0.0000

╶ Main iteration step ────────────────────────────────────────────────────────────────────────────────────────────────╴

    Iteration     │  Time  │  Dual cuts  │     Objective value     │   Objective gap   │     Current solution
     #: type      │  tot.  │   + | tot.  │       dual | primal     │    abs. | rel.    │    obj.fn. | max.err.
╶─────────────────┴────────┴─────────────┴─────────────────────────┴───────────────────┴──────────────────────────────╴

      | Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.5 LTS") 
      |  
      | CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2] 
      | Thread count: 20 physical cores, 20 logical processors, using up to 20 threads 
      |  
      | Non-default parameters: 
      | M

This short example already uses every ingredient you'll need. **The next sections slow down and look at each piece in more detail** (specifically more about objectives and constraints, what kind of terms you can use, and finally, retrieving solution details), before we put it all together in a larger and harder example.


## **4. Objectives**

Objectives specify what to minimize or maximize. Create an objective function by using:

`objective = SHOTpy.<Type>ObjectiveFunction(SHOTpy.ObjectiveDirection.<Direction>)`

- `<Type>`: `Linear`, `Quadratic`, or `Nonlinear`. 
- `<Direction>`: Either `Minimize` or `Maximize`.
- Add terms using: `objective.add()`.
- **Important:** Always call `problem.setObjective(objective)` after defining the objective.

Note, `NonlinearObjectiveFunction` supports any combination of linear (`SHOTpy.LinearTerm(coeff, var)`), quadratic (`SHOTpy.QuadraticTerm(coeff, var1, var2)`), and nonlinear terms (*e.g.*, `SHOTpy.sqrt(expr)`, `SHOTpy.exp(expr)`, `SHOTpy.log(expr)`, `SHOTpy.sin(expr)`, `SHOTpy.SignomialTerm(coeff, [(var, exp), ...])`, ...). You can also use `+`, `-`, `*`, `/`, `**`, etc.

### Linear Example:


In [ ]:
# Minimize: 2*x + 3*y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.LinearTerm(3.0, y))
#problem.setObjective(objective) 

### Quadratic Example:

In [ ]:
# Minimize: x^2 + y^2 + 2*x*y + x
objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))      # x^2
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))      # y^2
objective.add(SHOTpy.QuadraticTerm(2.0, x, y))      # 2*x*y 
objective.add(SHOTpy.LinearTerm(1.0, x))             # x
#problem.setObjective(objective) 

### Nonlinear Example:

In [ ]:
# Minimize: sqrt(x) + 2*exp(x) + log(y) + sin(x) + 1.0*x^2.5*y^-1.0 + 1.0*x^2 - 1.0
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.sqrt(x) + 2*SHOTpy.exp(x) + SHOTpy.log(y) + SHOTpy.sin(x) + 1.0*x**2.5*y**-1.0 + 1.0*x**2 - 1.0)
#problem.setObjective(objective) 

# Note, one could also model the objective like this:
# objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
# objective.add(SHOTpy.sqrt(x))
# objective.add(2*SHOTpy.exp(x))
# objective.add(SHOTpy.log(y))
# objective.add(SHOTpy.sin(x))
# objective.add(1.0*x**2 - 1.0)
# objective.add(SHOTpy.SignomialTerm(1.0, [(x, 2.5), (y, -1.0)]))


Note: The `problem.setObjective()` call is commented out because no problem object exists yet. 

## **5. Constraints**

Constraints are the rules, limits, and mathematical conditions that define the boundaries of your model. Create a constraint using:

`constraint = SHOTpy.<Type>Constraint(index, "name", lower_bound, upper_bound)`

- `<Type>`: `Linear`, `Quadratic`, or `Nonlinear`. 
- `index` (int): Unique internal index for the constraint.
- `"name"` (str): Human-readable name (e.g., "c1", "demand", "capacity").
- `lower_bound` (float): Minimum value for the left-hand side; use `SHOTpy.SHOT_DBL_MIN` if unbounded below.
- `upper_bound` (float): Maximum value for the left-hand side; use `SHOTpy.SHOT_DBL_MAX` if unbounded above.
- Add terms using: `constraint.add()`.
- **Important:** Always call `problem.addConstraint(constraint)` after defining the constraint.

Note, `NonlinearConstraint` supports any combination of linear (`SHOTpy.LinearTerm(coeff, var)`), quadratic (`SHOTpy.QuadraticTerm(coeff, var1, var2)`), and nonlinear terms (*e.g.*, `SHOTpy.sqrt(expr)`, `SHOTpy.exp(expr)`, `SHOTpy.log(expr)`, `SHOTpy.sin(expr)`, `SHOTpy.SignomialTerm(coeff, [(var, exp), ...])`, ...). You can also use `+`, `-`, `*`, `/`, `**`, etc.

### Linear Example:


In [7]:
# 2x + y <= 10
constraint = SHOTpy.LinearConstraint(0, "linear_con", SHOTpy.SHOT_DBL_MIN, 10.0)
constraint.add(SHOTpy.LinearTerm(2.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
#problem.addConstraint(constraint)

### Quadratic Example:

In [8]:
# x^2 + y^2 <= 4
constraint = SHOTpy.QuadraticConstraint(0, "quadratic_con", SHOTpy.SHOT_DBL_MIN, 4.0)
constraint.add(SHOTpy.QuadraticTerm(1.0, x, x))
constraint.add(SHOTpy.QuadraticTerm(1.0, y, y))
#problem.addConstraint(constraint)

### Nonlinear Example:

In [9]:
# sqrt(x) + 2*exp(x) + log(y) + sin(x) + 1.0*x^2.5*y^-1.0 + 1.0*x^2 <= 10.0
constraint = SHOTpy.NonlinearConstraint(0, "nonlinear_con", SHOTpy.SHOT_DBL_MIN, 10.0)
expr = SHOTpy.sqrt(x) + 2*SHOTpy.exp(x) + SHOTpy.log(y) + SHOTpy.sin(x) + 1.0*x**2.5*y**-1.0 + 1.0*x**2 + 1
constraint.add(expr)
#problem.addConstraint(constraint)

## **6. Retrieving Solution Details** 

To access the optimal solution, use `solver.getPrimalSolution()`, which returns a solution object with detailed information about the solve. Below you can find a table containing all available attributes:

| Attribute | Type | Description |
|-----------|------|-------------|
| `point` | List of floats | Vector containing the optimal values for all variables in order |
| `objValue` | Float | The objective function value at the optimal solution |
| `iterFound` | Integer | The iteration number at which this solution was found |
| `sourceType` | Enum | The type/source of the solution (e.g., from NLP solver, heuristic, etc.) |
| `sourceDescription` | String | Human-readable description of how the solution was obtained |
| `maxIntegerToleranceError` | Float | Maximum deviation of integer variables from their rounded values (before rounding was applied) |
| `maxDevatingConstraintLinear` | Pair (int, float) | Index and value of the linear constraint with largest violation |
| `maxDevatingConstraintQuadratic` | Pair (int, float) | Index and value of the quadratic constraint with largest violation |
| `maxDevatingConstraintNonlinear` | Pair (int, float) | Index and value of the nonlinear constraint with largest violation |
| `boundProjectionPerformed` | Boolean | Whether variable bounds were corrected (projected) to upper or lower bounds |
| `integerRoundingPerformed` | Boolean | Whether integer variables were rounded from their relaxed values |
| `displayed` | Boolean | Whether this solution has already been printed to console |

So far we've only used `sol.objValue` and `sol.point`, but as can be noted, the solution object carries a lot more than that. For example, the attributes let you inspect exactly how a solution was reached, which iteration found it, whether any rounding or bound projection was needed, and where the largest constraint violations (if any) occurred.

## **7. Putting It All Together: A Full MINLP Example** 

Let's combine everything to model and solve a complete mixed-integer nonlinear optimization problem.

Consider this mixed-integer nonlinear problem:

$$
\begin{align*}
\text{minimize} \quad & -x_1 - x_2\\
\text{subject to} \quad &  0.15(x_1 - 8)^2 + 0.1(x_2 - 6)^2 + 0.025e^{x_1}x_2^{-2} - 5 \leq 0 \\
& \frac{1}{x_1} + \frac{1}{x_2} - x_1^{0.5}x_2^{0.5} + 4 \leq 0 \\
& 2x_1 - 3x_2 \leq 2 \\
& 1 \leq x_1 \leq 20, \quad 1 \leq x_2 \leq 20 \\
& x_1 \in \mathbb{R}, x_2 \in \mathbb{Z} 
\end{align*}
$$

In [ ]:
# Create solver 
solver = SHOTpy.Solver()

# Get environment
env = solver.getEnvironment()

# Create problem
problem = SHOTpy.Problem(env)

# Variables
x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real, 1.0, 20.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Integer, 1.0, 20.0)
problem.addVariable(x1)
problem.addVariable(x2)

# Objective: minimize -x1 - x2 
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(-1.0, x1))
objective.add(SHOTpy.LinearTerm(-1.0, x2))
problem.setObjective(objective)

# Constraints
# 0.15(x1-8)^2 + 0.1(x2-6)^2 + 0.025*exp(x1)*x2^-2) - 5 <= 0
nonlin_con1 = SHOTpy.NonlinearConstraint(0, "nonlin_con1", SHOTpy.SHOT_DBL_MIN, 0.0)
nonlin_con1.add(0.15*(x1 - 8)**2 + 0.1*(x2 - 6)**2 + 0.025*SHOTpy.exp(x1)*x2**(-2) - 5.0)
problem.addConstraint(nonlin_con1)

# 1/x1 + 1/x2 - x1^0.5*x2^0.5 + 4 <= 0
nonlin_con2 = SHOTpy.NonlinearConstraint(1, "nonlin_con2", SHOTpy.SHOT_DBL_MIN, 0.0)
nonlin_con2.add(1.0/x1 + 1.0/x2 - x1**0.5 * x2**0.5 + 4.0)
problem.addConstraint(nonlin_con2)

# Linear constraint: 2*x1 - 3*x2 <= 2
lin_con = SHOTpy.LinearConstraint(2, "lin_con", SHOTpy.SHOT_DBL_MIN, 2.0)
lin_con.add(SHOTpy.LinearTerm(2.0, x1))
lin_con.add(SHOTpy.LinearTerm(-3.0, x2))
problem.addConstraint(lin_con)

# Finalize problem
problem.finalize()

# Set problem
solver.setProblem(problem)

# Solve
solver.solveProblem()


 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12

╶ Interior point search ──────────────────────────────────────────────────────────────────────────────────────────────╴

 Strategy selected:          cutting plane minimax

Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
    Iteration     │  Time  │    Cuts     │     Objective value     │  Objective diff.   
     #: type      │  tot.  │   + | tot.  │    problem | line srch  │    abs. | rel.    
╶─────────────────┴────────┴─────────────┴─────────────────────────┴──────────────────╴
     1: LP   

True

In [11]:
# Get detailed solution information
sol = solver.getPrimalSolution()

# Solution details
print(f"Objective Value: {sol.objValue}")
print(f"Iteration Found: {sol.iterFound}")
print(f"Source Description: {sol.sourceDescription}\n")

# Variable values
print("Variable Values:")
print(f"  x1 = {sol.point[0]}")
print(f"  x2 = {sol.point[1]}")

Objective Value: -20.90361501450038
Iteration Found: 8
Source Description: NLP fixed

Variable Values:
  x1 = 8.903615014500383
  x2 = 12.0


## **8. Key Takeaways**

### 1. **SHOTpy Uses a Consistent, Repeatable Workflow**
Every optimization problem follows the same seven-step pattern:
1. Create solver & environment.
2. Create problem.
3. Add variables.
4. Set objective.
5. Add constraints.
6. Finalize problem.
7. Solve and retrieve results.

Once you master this pattern, you can model any LP/QP/NLP/MILP/MI(QC)QP/MINLP problem with SHOTpy.

### 2. **Modeling in SHOTpy Is Straightforward and Intuitive**
You write mathematics almost exactly as you think it. Build expressions using natural Python operators and SHOTpy functions. Variables, objectives, and constraints all use the same intuitive pattern: create → add terms → register with problem.

### 3. **Solutions Contain Rich Information**
Beyond the optimal variable values, solutions reveal detailed information about the solve. This diagnostic information helps you understand solution quality and solver behavior.


### **Next Steps**
In **Notebook 2**, you'll learn more about the solver features. To name a few, you'll learn how to:
- **Read problems from a file**.
- **Configure the solver** for your specific needs.
- Use **callbacks** to customize SHOT's behavior.